In [25]:
import pandas as pd
original = "Project2026/data/original/diabetic_data.csv"
original_df = pd.read_csv(original)

In [26]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.feature_selection import SelectKBest, mutual_info_classif
import warnings
warnings.filterwarnings('ignore')


# ── 1. ICD-9 mapper ───────────────────────────────────────────────────────────
def map_icd9(code):
    try:
        code = str(code)
        if code.startswith('E') or code.startswith('V'):
            return 'external'
        code = int(code.split('.')[0])
        if   1   <= code <= 139: return 'infectious'
        elif 140 <= code <= 239: return 'neoplasms'
        elif 240 <= code <= 279: return 'endocrine'
        elif 280 <= code <= 289: return 'blood'
        elif 290 <= code <= 319: return 'mental'
        elif 320 <= code <= 389: return 'nervous'
        elif 390 <= code <= 459: return 'circulatory'
        elif 460 <= code <= 519: return 'respiratory'
        elif 520 <= code <= 579: return 'digestive'
        elif 580 <= code <= 629: return 'genitourinary'
        elif 630 <= code <= 679: return 'pregnancy'
        elif 680 <= code <= 709: return 'skin'
        elif 710 <= code <= 739: return 'musculoskeletal'
        elif 740 <= code <= 759: return 'congenital'
        elif 760 <= code <= 779: return 'perinatal'
        elif 780 <= code <= 799: return 'symptoms'
        elif 800 <= code <= 999: return 'injury'
        else: return 'other'
    except (ValueError, TypeError):
        return 'other'


# ── 2. Raw clean ──────────────────────────────────────────────────────────────
def raw_clean(df):
    df = df.copy()
    df.replace('?', np.nan, inplace=True)

    print(f"Rader innan cleaning: {len(df)}")

    # Ta bort patienter som dog eller skickades till hospice
    df = df[~df['discharge_disposition_id'].astype(str).isin(['11','19','20','21'])]
    print(f"Rader efter borttagning av hospice/deceased: {len(df)}")

    # Behåll bara första besöket per patient
    df = df.sort_values('encounter_id').drop_duplicates(subset='patient_nbr', keep='first')
    print(f"Rader efter deduplicering: {len(df)}")

    # Ta bort kolumner vi inte vill ha
    drop_cols = ["encounter_id", "patient_nbr", "weight", "payer_code", "medical_specialty"]
    df.drop(columns=drop_cols, inplace=True, errors='ignore')

    # Ta bort kända kolumner med extremt låg variation
    known_low_var = ['examide', 'citoglipton', 'troglitazone', 'acetohexamide']
    df.drop(columns=known_low_var, inplace=True, errors='ignore')

    # Mappa ICD-9
    for col in ['diag_1', 'diag_2', 'diag_3']:
        df[col] = df[col].astype(str).str.split('.').str[0].apply(map_icd9)

    # Feature engineering
    df['healthcare_utilization'] = (
        df['number_outpatient'] +
        df['number_emergency'] +
        df['number_inpatient']
    )

    med_cols = ['metformin','repaglinide','nateglinide','chlorpropamide',
                'glimepiride','glipizide','glyburide','pioglitazone',
                'rosiglitazone','insulin']
    existing_med_cols = [c for c in med_cols if c in df.columns]
    df['num_diabetes_meds'] = df[existing_med_cols].apply(
        lambda row: sum(v not in ['No', 'Steady'] for v in row), axis=1
    )

    print(f"Slutliga kolumner: {df.shape[1]}")
    return df


# ── 3. Ladda och förbered data ────────────────────────────────────────────────
original_df = pd.read_csv("Project2026/data/original/diabetic_data.csv")
cleaned = raw_clean(original_df)

print(f"\nKlassdistribution:\n{cleaned['readmitted'].value_counts()}")
print(f"\nMissingvärden kvar:\n{cleaned.isnull().sum()[cleaned.isnull().sum() > 0]}")

X = cleaned.drop("readmitted", axis=1)
y = cleaned["readmitted"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"\nTräning: {X_train.shape[0]} rader | Test: {X_test.shape[0]} rader")
print(f"Klassdistribution träning:\n{y_train.value_counts(normalize=True).round(3)}")


# ── 4. Kolumntyper ────────────────────────────────────────────────────────────
AGE_CATS = ['[0-10)','[10-20)','[20-30)','[30-40)','[40-50)',
            '[50-60)','[60-70)','[70-80)','[80-90)','[90-100)']
GLU_CATS = ['Norm', '>200', '>300']
A1C_CATS = ['Norm', '>7', '>8']

ORDINAL_COLS = ['age', 'max_glu_serum', 'A1Cresult']

NUMERIC_COLS = [
    'time_in_hospital', 'num_lab_procedures', 'num_procedures',
    'num_medications', 'number_outpatient', 'number_emergency',
    'number_inpatient', 'number_diagnoses',
    'healthcare_utilization', 'num_diabetes_meds'
]

CANDIDATE_NOMINAL = [
    'race', 'gender', 'admission_type_id',
    'discharge_disposition_id', 'admission_source_id',
    'change', 'diabetesMed',
    'diag_1', 'diag_2', 'diag_3',
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose',
    'miglitol', 'tolazamide', 'insulin',
    'glyburide-metformin', 'glipizide-metformin',
    'glimepiride-pioglitazone', 'metformin-rosiglitazone',
    'metformin-pioglitazone'
]

# Behåll bara kolumner som faktiskt finns kvar efter cleaning
NOMINAL_COLS = [c for c in CANDIDATE_NOMINAL if c in X_train.columns]
NUMERIC_COLS  = [c for c in NUMERIC_COLS      if c in X_train.columns]
ORDINAL_COLS  = [c for c in ORDINAL_COLS       if c in X_train.columns]
ORDINAL_CATS  = [cats for col, cats in zip(
    ['age', 'max_glu_serum', 'A1Cresult'],
    [AGE_CATS, GLU_CATS, A1C_CATS]
) if col in ORDINAL_COLS]

print(f"\nNominala kolumner  ({len(NOMINAL_COLS)}): {NOMINAL_COLS}")
print(f"Numeriska kolumner ({len(NUMERIC_COLS)}): {NUMERIC_COLS}")
print(f"Ordinala kolumner  ({len(ORDINAL_COLS)}): {ORDINAL_COLS}")


# ── 5. Preprocessor ───────────────────────────────────────────────────────────
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True)),
            ('scaler',  StandardScaler()),
        ]), NUMERIC_COLS),

        ('ord', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OrdinalEncoder(
                categories=ORDINAL_CATS,
                handle_unknown='use_encoded_value',
                unknown_value=-1
            )),
        ]), ORDINAL_COLS),

        ('nom', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(
                drop='first',
                sparse_output=False,
                handle_unknown='ignore'
            )),
        ]), NOMINAL_COLS),
    ],
    remainder='drop'
)


# ── 6. Modeller ───────────────────────────────────────────────────────────────
DEV_MODE = True  # sätt till False för final körning

models = {
    "Linear SVC (manuella weights)": Pipeline([
        ('pre', preprocessor),
        ('selector', SelectKBest(score_func=mutual_info_classif, k=10)),  # prova 30, 50, 100
        ('clf', LinearSVC(
            dual=False,
            class_weight={'<30': 8, '>30': 2, 'NO': 1},
            max_iter=2000, random_state=42,
            
        )),
    ])#,

    # "Linear SVC + SMOTE": ImbPipeline([
    #     ('pre', preprocessor),
    #     ('smote', SMOTE(random_state=42)),
    #     ('selector', SelectKBest(score_func=mutual_info_classif, k=20)),  # prova 30, 50, 100
    #     ('clf', LinearSVC(
    #         dual=False,
    #         class_weight=None,
    #         max_iter=2000, random_state=42
    #     )),
    # ]),

    # "Random Forest + SMOTE": ImbPipeline([
    #     ('pre', preprocessor),
    #     ('smote', SMOTE(random_state=42)),
    #     ('selector', SelectKBest(score_func=mutual_info_classif, k=20)),  # prova 30, 50, 100
    #     ('clf', RandomForestClassifier(
    #         n_estimators=100 if DEV_MODE else 300,
    #         max_depth=20,
    #         n_jobs=1,
    #         random_state=42
    #     )),
    # ]),
}


# ── 7. Träning & utvärdering ──────────────────────────────────────────────────
cv = StratifiedKFold(
    n_splits=3 if DEV_MODE else 5,
    shuffle=True,
    random_state=42
)

SCORE_METRICS = [
    'accuracy', 'balanced_accuracy',
    'f1_macro', 'precision_macro', 'recall_macro'
]

for name, pipe in models.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print('='*50)

    cv_results = cross_validate(
        pipe, X_train, y_train,
        cv=cv, scoring=SCORE_METRICS,
        n_jobs=1
    )
    for metric in SCORE_METRICS:
        scores = cv_results[f'test_{metric}']
        print(f"  CV {metric:<25} {scores.mean():.4f} ± {scores.std():.4f}")

    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    print(f"\n  Testset resultat:")
    print(classification_report(y_test, y_pred, zero_division=0))
    print("  Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))

Rader innan cleaning: 101766
Rader efter borttagning av hospice/deceased: 100114
Rader efter deduplicering: 70439
Slutliga kolumner: 43

Klassdistribution:
readmitted
NO     41901
>30    22242
<30     6296
Name: count, dtype: int64

Missingvärden kvar:
race              1922
max_glu_serum    67054
A1Cresult        57561
dtype: int64

Träning: 56351 rader | Test: 14088 rader
Klassdistribution träning:
readmitted
NO     0.595
>30    0.316
<30    0.089
Name: proportion, dtype: float64

Nominala kolumner  (29): ['race', 'gender', 'admission_type_id', 'discharge_disposition_id', 'admission_source_id', 'change', 'diabetesMed', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'tolazamide', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone']
Numeriska kolum

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report
import itertools

k_values = [10, 20, 30, 50, 75, 100, 125, 150, 159]
# Olika kombinationer av class weights för <30, >30, NO
weight_combos = [
    {'<30': 4, '>30': 2, 'NO': 1},
    {'<30': 6, '>30': 2, 'NO': 1},
    {'<30': 8, '>30': 2, 'NO': 1},
    {'<30': 10, '>30': 2, 'NO': 1},
    {'<30': 8, '>30': 3, 'NO': 1},
    {'<30': 8, '>30': 1, 'NO': 1},
    {'<30': 12, '>30': 3, 'NO': 1},
    'balanced',
    None,
]

results = []

total = len(k_values) * len(weight_combos)
i = 0

for k, weights in itertools.product(k_values, weight_combos):
    i += 1
    print(f"  [{i}/{total}] k={k}, weights={weights}", end='\r')

    pipe_k = Pipeline([
        ('pre', preprocessor),
        ('selector', SelectKBest(score_func=f_classif, k=k)),
        ('clf', LinearSVC(
            dual=False,
            class_weight=weights,
            max_iter=2000, random_state=42
        )),
    ])

    pipe_k.fit(X_train, y_train)
    y_pred = pipe_k.predict(X_test)

    report = classification_report(y_test, y_pred, output_dict=True, zero_division=0)
    results.append({
        'k':            k,
        'weights':      str(weights),
        'balanced_acc': balanced_accuracy_score(y_test, y_pred),
        'f1_macro':     f1_score(y_test, y_pred, average='macro', zero_division=0),
        '<30_recall':   report['<30']['recall'],
        '<30_f1':       report['<30']['f1-score'],
        '<30_precision':report['<30']['precision'],
    })

results_df = pd.DataFrame(results).sort_values('balanced_acc', ascending=False)

print("\n\nTop 10 kombinationer (sorterat på balanced_acc):")
print(results_df.head(10).to_string(index=False))

print(f"\nBästa k + weights (balanced_acc): k={results_df.iloc[0]['k']}, weights={results_df.iloc[0]['weights']}")
print(f"Bästa k + weights (f1_macro):     k={results_df.sort_values('f1_macro', ascending=False).iloc[0][['k','weights']].to_dict()}")
print(f"Bästa k + weights (<30 recall):   k={results_df.sort_values('<30_recall', ascending=False).iloc[0][['k','weights']].to_dict()}")